In [2]:
import os
base = os.path.expanduser("/Users/muhdfariq/Library/CloudStorage/GoogleDrive-fariqdude@gmail.com/.shortcut-targets-by-id/1Wir3GBSMVsYy62wAWvbgtt6iFQgpUOLj/Esports_project")


In [3]:
for f in os.listdir("/Users/muhdfariq/Library/CloudStorage/GoogleDrive-fariqdude@gmail.com/.shortcut-targets-by-id/1Wir3GBSMVsYy62wAWvbgtt6iFQgpUOLj/Esports_project"):
  print (f)

MemberB(Liam).ipynb
calibration_curve.png
encoders.pkl
Esports_Predictor.ipynb
Dataset
calibrated_model.pkl
feature_cols.pkl
Parquets


In [4]:
import pickle
import polars as pl
import pandas as pd
import numpy as np
from itertools import combinations

# Set the base path to your local Google Drive shortcut target
base = "/Users/muhdfariq/Library/CloudStorage/GoogleDrive-fariqdude@gmail.com/.shortcut-targets-by-id/1Wir3GBSMVsYy62wAWvbgtt6iFQgpUOLj/Esports_project"

# 1. Load the model and dictionaries
with open(f"{base}/calibrated_model.pkl", "rb") as f:
    calibrated_model = pickle.load(f)
    
with open(f"{base}/feature_cols.pkl", "rb") as f:
    feature_cols = pickle.load(f)
    
with open(f"{base}/encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

# 2. Load the parquets
final_draft = pl.read_parquet(f"{base}/Parquets/final_draft.parquet")
champ_wr = pl.read_parquet(f"{base}/Parquets/champ_winrates.parquet")
pairs = pl.read_parquet(f"{base}/Parquets/champ_synergies.parquet")

print("Files successfully loaded! Ready for feature engineering.")

Files successfully loaded! Ready for feature engineering.


/Users/muhdfariq/miniconda3/envs/wif3009/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/muhdfariq/miniconda3/envs/wif3009/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/muhdfariq/miniconda3/envs/wif3009/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Cali

In [5]:
# Convert Polars dataframe to Pandas for feature engineering
df = final_draft.to_pandas()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# ---------------------------------------------------------
# LAYER 1: Win Rate Features
# ---------------------------------------------------------
wr_map = dict(zip(champ_wr["champion"].to_list(), champ_wr["win_rate"].to_list()))
global_avg_wr = champ_wr["win_rate"].mean()

role_cols = ["blue_top", "blue_jng", "blue_mid", "blue_bot", "blue_sup",
             "red_top", "red_jng", "red_mid", "red_bot", "red_sup"]

for col in role_cols:
    df[f"{col}_wr"] = df[col].map(wr_map).fillna(global_avg_wr)

# ---------------------------------------------------------
# LAYER 2: Synergy Scores
# ---------------------------------------------------------
pair_map = {}
for row in pairs.to_dicts():
    key = tuple(sorted([row["champion"], row["champ2"]]))
    pair_map[key] = row["pair_win_rate"]

def team_synergy_score(champs):
    scores = []
    valid = [c for c in champs if pd.notna(c)]
    for c1, c2 in combinations(valid, 2):
        key = tuple(sorted([c1, c2]))
        if key in pair_map:
            scores.append(pair_map[key])
    return np.mean(scores) if scores else 0.5

blue_roles = ["blue_top", "blue_jng", "blue_mid", "blue_bot", "blue_sup"]
red_roles  = ["red_top", "red_jng", "red_mid", "red_bot", "red_sup"]

df["blue_synergy"] = df[blue_roles].apply(lambda row: team_synergy_score(row.tolist()), axis=1)
df["red_synergy"]  = df[red_roles].apply(lambda row: team_synergy_score(row.tolist()), axis=1)

# ---------------------------------------------------------
# LAYER 3: Aggregate Features & Differentials
# ---------------------------------------------------------
df["blue_team_avg_wr"] = df[[f"{r}_wr" for r in blue_roles]].mean(axis=1)
df["red_team_avg_wr"]  = df[[f"{r}_wr" for r in red_roles]].mean(axis=1)
df["wr_diff"]          = df["blue_team_avg_wr"] - df["red_team_avg_wr"]
df["synergy_diff"]     = df["blue_synergy"] - df["red_synergy"]

# ---------------------------------------------------------
# LAYER 4: Encode Categoricals
# ---------------------------------------------------------
cat_cols = role_cols + ["patch", "league", "blue_team", "red_team"]

for col in cat_cols:
    le = encoders[col]
    
    def safe_encode(val):
        # 1. If the model saw this exact value during training, encode it normally
        if val in le.classes_:
            return le.transform([val])[0]
        # 2. If the model explicitly learned an 'UNKNOWN' fallback, use it
        elif "UNKNOWN" in le.classes_:
            return le.transform(["UNKNOWN"])[0]
        # 3. Otherwise, use -1 so LightGBM knows it's an unseen category
        else:
            return -1
            
    df[col + "_enc"] = df[col].apply(safe_encode)

# ---------------------------------------------------------
# FINAL SPLIT: Recreate the 20% Holdout Test Set
# ---------------------------------------------------------
cutoff = int(len(df) * 0.8)
X_test = df.iloc[cutoff:][feature_cols]

print(f"Feature matrix rebuilt successfully! Shape: {X_test.shape}")

Feature matrix rebuilt successfully! Shape: (919, 28)


In [7]:
import shap
import matplotlib.pyplot as plt

print("Extracting raw LightGBM model from calibrator wrapper...")
# 1. Bypass the CalibratedClassifierCV shell to get the raw model
lgbm_model = calibrated_model.calibrated_classifiers_[0].estimator

print("Initializing TreeExplainer (this might take a few seconds)...")
# 2. Initialize the ultra-fast TreeExplainer
explainer = shap.TreeExplainer(lgbm_model)

print("Calculating global SHAP values (Bypassing Pandas metadata check)...")
# 3. Compute SHAP values using the raw NumPy array (.values) to avoid LightGBM crashes
shap_values = explainer(X_test.values)

# 4. Re-attach the feature names so SHAP knows what to label on the plots
shap_values.feature_names = feature_cols

print("Generating visual proofs...")
# 5. Generate and save the Beeswarm Plot
shap.plots.beeswarm(shap_values, show=False)
plt.savefig(f"{base}/beeswarm_meta.png", bbox_inches='tight')
plt.clf() 

# 6. Generate and save the Bar Plot
shap.plots.bar(shap_values, show=False)
plt.savefig(f"{base}/bar_meta.png", bbox_inches='tight')
plt.clf()

print(f"Success! SHAP values calculated for {shap_values.shape[0]} matches.")
print("Global plots saved to your shared Drive folder.")

Extracting raw LightGBM model from calibrator wrapper...
Initializing TreeExplainer (this might take a few seconds)...
Calculating global SHAP values (Bypassing Pandas metadata check)...
Generating visual proofs...
Success! SHAP values calculated for 919 matches.
Global plots saved to your shared Drive folder.


<Figure size 800x650 with 0 Axes>

In [8]:
def explain_single_draft(match_index=0):
    """
    Simulates what happens when a user clicks 'Predict' in the UI.
    Generates a local waterfall plot and a text breakdown for the Agent.
    """
    print(f"--- Explaining Match Index {match_index} ---")
    
    # 1. Generate visual for Member D (UI)
    shap.plots.waterfall(shap_values[match_index], show=False)
    plt.savefig(f"{base}/current_draft_waterfall.png", bbox_inches='tight')
    plt.clf()
    print("Local Waterfall plot saved for UI.")
    
    # 2. Package text data for Member E's LangGraph node
    single_row = X_test.iloc[match_index]
    local_shap = shap_values[match_index]
    
    impacts = []
    # Zip the feature names, actual draft values, and SHAP math together
    for col, val, shap_val in zip(feature_cols, single_row, local_shap.values):
        if abs(shap_val) > 0.01: # Filter out noise (only keep big impacts)
            impacts.append({
                "feature": col,
                "value": float(val),
                "impact_on_win_prob": float(shap_val)
            })
    
    # Sort by biggest impact (positive or negative)
    impacts = sorted(impacts, key=lambda x: abs(x["impact_on_win_prob"]), reverse=True)
    
    agent_payload = {
        "expected_base_win_prob": float(local_shap.base_values),
        "top_drivers": impacts[:5] # Only give the Agent the Top 5 reasons
    }
    
    return agent_payload
    
# Test the function on the very first match in your test set
draft_breakdown = explain_single_draft(0)

print("\nPayload ready for Member E's Agent:")
import json
print(json.dumps(draft_breakdown, indent=2))

--- Explaining Match Index 0 ---
Local Waterfall plot saved for UI.

Payload ready for Member E's Agent:
{
  "expected_base_win_prob": 0.28706370151165156,
  "top_drivers": [
    {
      "feature": "league_enc",
      "value": 30.0,
      "impact_on_win_prob": -0.597333995649049
    },
    {
      "feature": "synergy_diff",
      "value": -0.006609827896908116,
      "impact_on_win_prob": -0.5661362322656045
    },
    {
      "feature": "red_sup_enc",
      "value": 30.0,
      "impact_on_win_prob": 0.3549217769829433
    },
    {
      "feature": "blue_bot_wr",
      "value": 0.514003294892916,
      "impact_on_win_prob": -0.3004012123967691
    },
    {
      "feature": "blue_synergy",
      "value": 0.5150307087795167,
      "impact_on_win_prob": 0.2825454819409092
    }
  ]
}


In [9]:
print("Packaging SHAP output for Member E...")

shap_output = {
    "values": shap_values.values,     # The raw SHAP math
    "features": feature_cols,         # The column names
    "data": X_test.values             # The actual match data
}

with open(f"{base}/shap_output.pkl", "wb") as f:
    pickle.dump(shap_output, f)

print(f"Success! shap_output.pkl saved to Drive.")
print("Member E is now fully unblocked to start building the Agent!")

Packaging SHAP output for Member E...
Success! shap_output.pkl saved to Drive.
Member E is now fully unblocked to start building the Agent!


AttributeError: 'dict' object has no attribute 'pkl'